## Common Challenges with LLMs

LLMs face notable limitations in several key areas:

1. Knowledge Cutoff
- Unable to access real-time or recent information beyond training data
- Cannot browse the internet or access current events unless explicit arrangement are done

2. Hallucinations
- May generate plausible-sounding but factually incorrect information
- Can confidently assert false statements as truth

3. Private & Sensitive Data
- Cant respond correctly for proprietary, or personal information

> In order to address these limitations, RAG concept is introduced. Empowering LLMs to access and utilize external data sources.

---

## Why can't I just finetune the LLM with th the external knowledge base?

1. Training is slow and expensive
2. Frequent updates to the knowledge base means retraining the LLM, more money
3. Technical expertise in the organization to handle training and decision making

---

## Alternate ways: [In-context learning](https://arxiv.org/pdf/2507.16003)

It is a core capability of LLM models like GPT-3 that allows the model to learn from the context it is given. In this mode the model simply learns from the examples given in the prompt-without updating its parameters.

One of such techniques: few shot learning

`Few-shot learning` is when you give an LLM a few examples of the desired input→output pattern in the prompt, so it understands the task without any fine-tuning.

Example — Sentiment Classification:
```
Classify the sentiment of the sentence as Positive or Negative.

Sentence: "The movie was absolutely fantastic!"
Sentiment: Positive

Sentence: "I hated every minute of it."
Sentiment: Negative

Sentence: "The product exceeded all my expectations."
Sentiment: Positive

Sentence: "Worst customer service I've ever experienced."
Sentiment: ?
```

Variants:

|Type	   | Examples given|
|----------|---------------|
|Zero-shot|	0 — just a task description|
|One-shot|	1 example|
|Few-shot|	2–5+ examples|

> Instead of few shot prompting, if you give full context/data required to solve a task along with your prompt, It's called RAG.

---

## RAG (Retrieve, Augment, and Generate)

RAG is a technique that leverages the power of LLMs to enhance their capabilities by integrating the ability to retrieve, augment, and generate information from external sources. This approach allows LLMs to access a broader range of information, including real time data, and generate more accurate and context-aware responses. RAG is particularly useful in scenarios where LLMs are tasked with generating responses based on a limited amount of training data, such as in the context of knowledge cutoff or privacy concerns. It is beasically a way to make a LLM smarter by giving it extra information at the time you ask your questions and get contextual relevant response.

```
 User Query ──┐
              ├──► [ Prompt ] ──► [ LLM ] ──► Response
 Context   ──┘
```

> **Analogy:** Just like an engineer uses their degree (LLM training) + on-the-job reference material (retrieved docs) to solve real problems — RAG equips an LLM with the right context at the right time.

---

RAG can be divided in 4 steps:

1. **Indexing** - Preparing the external knowledge base
2. **Retrieving** - Understand the query and consulting the external knowledge base to identify the chunks that helps in responding the query. Finally building the context based on those chunks.
3. **Augmenting** - Generate a prompt using [query, external context]
4. **Generating** - LLM prepares response using In-context learning

### Overview
```
┌─────────┐                    ┌─────────────┐
│  query  │                    │  Documents  │
└────┬────┘                    └──────┬──────┘
     │                                │
     │                    ┌───────────┼───────────┐
     │                    ▼           ▼           ▼
     │              ┌─────────┐ ┌─────────┐ ┌─────────┐
     │              │ Chunk 1 │ │ Chunk 2 │ │ Chunk 3 │
     │              └────┬────┘ └────┬────┘ └────┬────┘
     │                   └───────────┼────────────┘
     │                               ▼
     │                   ┌───────────────────────┐
     └──────────────────►│     Embedding LLM     │
                         └───────────┬───────────┘
          ┌────────────────┐         │
          │   Embedding    │◄────────┘
          └───────┬────────┘         │
                  │         ┌────────┴────────────────┐
                  │         ▼         ▼               ▼
                  │    ┌─────────┐ ┌─────────┐ ┌─────────┐
                  │    │  Emb 1  │ │  Emb 2  │ │  Emb 3  │
                  │    └────┬────┘ └────┬────┘ └────┬────┘
                  │         └───────────┼────────────┘
                  │                     ▼
                  │           ┌──────────────────┐
                  └──────────►│   Most similar   │
                              └────────┬─────────┘
                                       ▼
                              ┌──────────────────┐
                              │     Gen. LLM     │
                              └────────┬─────────┘
                                       ▼
                                   response

```

### Naive RAG (foundational)
```
                        ╔══════════════════════════════════════════════╗
                        ║              [INDEXING]                      ║
                        ║                                              ║
                        ║   ┌──────────────────┐                       ║
                        ║   │       docs       │                       ║
                        ║   └────────┬─────────┘                       ║
                        ║            │  Parsing + preprocessing        ║
                        ║            ▼                                 ║
                        ║   ┌──────────────────┐                       ║
                        ║   │   Parsed docs    │                       ║
                        ║   └────────┬─────────┘                       ║
                        ║            │  Chunking                       ║
                        ║            ▼                                 ║
                        ║   ┌──────────────────┐   ┌────────────────┐  ║
                        ║   │     chunks       ├──►│ Embedding model│  ║
                        ║   └──────────────────┘   └───────┬────────┘  ║
                        ║                                  │ Vectorize ║
                        ║                          ┌───────▼─────────┐ ║
                        ║                          │ [■ ■ ■ ■ ■ ■]   │ ║
                        ║                          └────────┬────────┘ ║
                        ║                                   │ Indexing ║
                        ║                          ┌────────▼────────┐ ║
                        ║                     ┌───►┤  Vector store   ├─╫──────────────────┐
                        ║                     |    └─────────────────┘ ║  Retrieve        │
                        ╚═════════════════════|════════════════════════╝                  │
                                              |                                           ▼
user ──► query ──► Embedding model ──► vectorize                           ┌──────────────────────┐
  ▲                                                                        │      [AUGMENT]       │
  │                                                                        │  ┌────────────────┐  │
  │                                                                        │  │     prompt     │  │
  │                                                                        │  ├────────────────┤  │
  │                                                                        │  │  Relevant docs │  │
  │                                                                        │  ├────────────────┤  │
  │                                                                        │  │     query      │  │
  │                                                                        └──────────┬───────────┘
  │                                                                                   │
  │                        Generate                                                   ▼
  └──── response ◄─────────────────────────────────────────────────────── Gen. LLM ◄──┘

```